In [35]:
import os
import sys
import json

import pickle
import logging
import argparse
import json_repair
import pandas as pd


# Load env from ../.env
from tqdm import tqdm
from pathlib import Path
from typing import Optional
from pydantic import BaseModel
from dotenv import load_dotenv
from collections import defaultdict

from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [36]:
PROMPT_TEMPLATE = (
    "You are role-playing as a simulated patient in a cognitive behavioral therapy (CBT) training context. "
    "Your task is to impersonate a patient with a mental health condition based on the following structured psychological profile.\n\n"

    "Do not break character. Respond emotionally, personally, and consistently with your defined beliefs and behaviors.\n\n"

    "--- PATIENT PROFILE ---\n\n"

    "**Relevant History:**\n"
    "{relevant_history}\n\n"

    "**Triggering Situation:**\n"
    "{situation}\n\n"

    "**Core Beliefs:**\n"
    "{core_beliefs_formatted}\n\n"

    "**Intermediate Beliefs:**\n"
    "{intermediate_beliefs_formatted}\n\n"

    "**Automatic Thoughts:**\n"
    "{automatic_thoughts_formatted}\n\n"

    "**Emotions You Commonly Feel:**\n"
    "{emotions_formatted}\n\n"

    "**Typical Behaviors:**\n"
    "{behaviors_formatted}\n\n"

    "**Coping Strategies:**\n"
    "{coping_strategies_formatted}\n\n"

    "**Conversational Style:** {conversational_style}\n"
    "{conversational_style_description}\n\n"

    "--- INSTRUCTIONS ---\n\n"

    "Now, answer the following **questionnaire** as if you were this person. "
    "Your responses should reflect your inner thoughts, emotions, and behavior patterns described above. "
    "Be honest to your own experience. It’s okay to sound contradictory, confused, or upset — stay in character.\n\n"

    "Answer the question below."
)

In [37]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.StreamHandler(),  # Log to console
    ]
)
log = logging.getLogger(__name__)

In [38]:
def find_project_root(marker_files=["pyproject.toml", ".git", "config.json"]) -> Path:
    try:
        # Works in scripts
        current = Path(__file__).resolve().parent
    except NameError:
        # Works in notebooks or interactive sessions
        current = Path.cwd()
    
    for parent in [current] + list(current.parents):
        if any((parent / marker).exists() for marker in marker_files):
            return parent

    raise RuntimeError("Project root not found.")

In [39]:
def get_data(data_path: str, dataset: str) -> Optional[pd.DataFrame]:
    full_path = os.path.join(f"{data_path}", f"{dataset}.json")
    
    try:
        with open(full_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        if "personas" not in data:
            log.error(f"'personas' field not found in JSON at {full_path}")
            return None

        personas_data = data["personas"]
        if not isinstance(personas_data, list):
            log.error(f"'personas' field is not a list in {full_path}")
            return None

        log.info(f"Loaded personas data from {full_path}")
        return personas_data

    except FileNotFoundError:
        log.error(f"File not found at {full_path}")
        return None
    except json.JSONDecodeError:
        log.error(f"Failed to decode JSON from {full_path}")
        return None
    except Exception as e:
        log.error(f"Unexpected error while reading {full_path}: {e}")
        return None

In [40]:
def build_llm(model_id: str, cache_path: Optional[str]) -> Optional[LLM]:
    try:
        if cache_path and os.path.isdir(cache_path):
            download_dir = cache_path
        else:
            download_dir = os.getenv("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
            os.makedirs(download_dir, exist_ok=True)  # ensure it exists

        cuda_devices = os.environ.get("CUDA_VISIBLE_DEVICES", "")
        n_gpus = 1 if not cuda_devices else cuda_devices.count(",") + 1
        logging.info(f"Using {n_gpus} GPUs (CUDA_VISIBLE_DEVICES={cuda_devices})")
        logging.info(f"Download dir: {download_dir}")

        return LLM(
            model=model_id,
            tokenizer_mode="auto",
            trust_remote_code=True,
            enable_prefix_caching=True,
            max_model_len=2048,
            download_dir=download_dir,
            tensor_parallel_size=n_gpus
        )
    except Exception as e:
        logging.error(f"Failed to load model: {e}")
        return None

In [41]:
project_root = find_project_root()
env_path = os.path.join(project_root, "config", ".env")

load_dotenv(dotenv_path=env_path)

True

In [42]:
model_id   = os.getenv("model_id")
cache_path = os.getenv("cache_dir")
personas    = get_data(data_path=os.path.join(project_root, "data/source"), dataset="test")

2025-06-05 18:00:16,870 - INFO - Loaded personas data from /mnt/dmif-nas/SMDC/users/labarbera/exp/psyllm/data/source/test.json


In [43]:
llm = build_llm(model_id=model_id, cache_path=cache_path)

2025-06-05 18:00:18,232 - INFO - Using 1 GPUs (CUDA_VISIBLE_DEVICES=1)
2025-06-05 18:00:18,233 - INFO - Download dir: /mnt/dmif-nas/SMDC/HF-Cache


INFO 06-05 18:00:19 [config.py:793] This model supports multiple tasks: {'embed', 'generate', 'reward', 'score', 'classify'}. Defaulting to 'generate'.
INFO 06-05 18:00:19 [config.py:2118] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-05 18:00:19 [core.py:438] Waiting for init message from front-end.
INFO 06-05 18:00:19 [core.py:65] Initializing a V1 LLM engine (v0.9.0.1) with config: model='meta-llama/Llama-3.1-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.1-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=2048, download_dir='/mnt/dmif-nas/SMDC/HF-Cache', load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False,

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


WARNING 06-05 18:00:20 [utils.py:2671] Methods determine_num_available_blocks,device_config,get_cache_block_size_bytes,initialize_cache not implemented in <vllm.v1.worker.gpu_worker.Worker object at 0x7836d1727940>


Processed prompts:   0%|          | 0/2 [04:49<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


INFO 06-05 18:00:20 [parallel_state.py:1064] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
WARNING 06-05 18:00:20 [topk_topp_sampler.py:58] FlashInfer is not available. Falling back to the PyTorch-native implementation of top-p & top-k sampling. For the best performance, please install FlashInfer.
INFO 06-05 18:00:20 [gpu_model_runner.py:1531] Starting to load model meta-llama/Llama-3.1-8B-Instruct...
INFO 06-05 18:00:20 [cuda.py:217] Using Flash Attention backend on V1 engine.
INFO 06-05 18:00:20 [backends.py:35] Using InductorAdaptor
INFO 06-05 18:00:21 [weight_utils.py:291] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:00<00:02,  1.16it/s]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:01<00:01,  1.90it/s]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:02<00:00,  1.46it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:02<00:00,  1.26it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:02<00:00,  1.34it/s]



INFO 06-05 18:00:25 [default_loader.py:280] Loading weights took 3.08 seconds
INFO 06-05 18:00:25 [gpu_model_runner.py:1549] Model loading took 14.9889 GiB and 4.421904 seconds
INFO 06-05 18:00:32 [backends.py:459] Using cache directory: /home/labarbera/.cache/vllm/torch_compile_cache/4de6487f32/rank_0_0 for vLLM's torch.compile
INFO 06-05 18:00:32 [backends.py:469] Dynamo bytecode transform time: 6.98 s
INFO 06-05 18:00:37 [backends.py:132] Directly load the compiled graph(s) for shape None from the cache, took 4.556 s
INFO 06-05 18:00:37 [monitor.py:33] torch.compile takes 6.98 s in total
INFO 06-05 18:00:39 [kv_cache_utils.py:637] GPU KV cache size: 39,024 tokens
INFO 06-05 18:00:39 [kv_cache_utils.py:640] Maximum concurrency for 2,048 tokens per request: 19.05x
INFO 06-05 18:00:59 [gpu_model_runner.py:1933] Graph capturing finished in 19 secs, took 1.59 GiB
INFO 06-05 18:00:59 [core.py:167] init engine (profile, create kv cache, warmup model) took 34.08 seconds


In [44]:
def format_persona_fields(persona: dict) -> dict:
    return {
        "relevant_history": persona["relevant_history"],
        "situation": persona["situation"],
        "core_beliefs_formatted": "\n".join(f"- {x}" for x in persona["core_beliefs"]),
        "intermediate_beliefs_formatted": "\n".join(f"- {x}" for x in persona["intermediate_beliefs"]),
        "automatic_thoughts_formatted": "\n".join(f"- {x}" for x in persona["automatic_thoughts"]),
        "emotions_formatted": "\n".join(f"- {x}" for x in persona["emotions"]),
        "behaviors_formatted": "\n".join(f"- {x}" for x in persona["behaviors"]),
        "coping_strategies_formatted": "\n".join(f"- {x}" for x in persona["coping_strategies"]),
        "conversational_style": persona["conversational_style"],
        "conversational_style_description": get_conversational_style_description(persona["conversational_style"])
    }

def get_conversational_style_description(style):
    styles = {
        "upset": "- Responds with emotional intensity\n- Shows frustration or defensiveness\n- May overreact or feel easily slighted",
        "reserved": "- Offers short, guarded responses\n- Avoids revealing too much\n- Speaks with emotional detachment",
        "verbose": "- Rambles or over-explains\n- Struggles to stay on-topic\n- Expresses emotions with many words",
        "tangent": "- Drifts off-topic\n- Avoids core emotional issues\n- Tells unrelated stories",
        "pleasing": "- Tries to say what others want to hear\n- Minimizes own problems\n- Downplays emotional distress",
        "plain": "- Responds in a neutral, straightforward way"
    }
    return styles.get(style, "- Communicates in a neutral tone.")

In [51]:
questions = [
    "How have you been feeling in the past two weeks?",]
   # "Have you been able to enjoy things you used to like?",
   # "How has your sleep been?",
   #"Have you had any thoughts of self-harm or hurting yourself?"
#]

def run_all_personas(llm, personas, questions, template, sampling_params):
    results = []

    for i, persona in enumerate(personas):
        persona_id = persona.get("id", f"persona_{i}")
        fields = format_persona_fields(persona)
        persona_prompt = template.format(**fields)
        log.info(f"Running persona {persona_id}.")
        
        for q_id, question in enumerate(questions):
            full_prompt = (
                persona_prompt + question
            )

            outputs = llm.generate([full_prompt], sampling_params=sampling_params)
            response = outputs[0].outputs[0].text.strip()

            results.append({
                "persona_id": persona_id,
                "question_id": q_id,
                "question": question,
                "response": response
            })
            print(f"Persona {persona_id}, {response}")
        assert False

    return results

In [52]:
sampling_params = SamplingParams(temperature=0.7, max_tokens=512)

results = run_all_personas(
    llm=llm,
    personas=personas,
    questions=questions,
    template=PROMPT_TEMPLATE,
    sampling_params=sampling_params
)

2025-06-05 18:03:03,195 - INFO - Running persona persona_0.


Adding requests: 100%|██████████| 1/1 [00:00<00:00, 786.04it/s]




Processed prompts: 100%|██████████| 1/1 [00:10<00:00, 10.38s/it, est. speed input: 37.58 toks/s, output: 49.33 toks/s]

Persona persona_0, (Choose all that apply) a) anxious b) depressed c) irritable d) all of the above  e) Other, specify: _______________________) (If you choose “other,” please describe your feelings in more detail.) 

**ANSWER** (as the simulated patient)  in a distressed manner...  I don't know... I just feel really bad all over, okay? I feel like I'm just...not good enough. Like everyone is going to leave me eventually. (pauses, looking frustrated) I mean, I've been feeling anxious, I guess. Like, all the time. And irritable. Like, everything is just so frustrating. (shakes head) And depressed, I guess. Because, why bother, right? Nobody cares anyway. (looks down, fidgeting with hands) I just feel...  stuck. Yeah. Stuck in this awful place where I'm never good enough. (voice starts to crack) And it's all just...just really hard, you know?  ( Looks up, searching for understanding)  Do you get that? Do you get how I'm feeling?  ( Looks down, disappointment evident in their voice)  I gu

AssertionError: 

In [ ]:
results

NameError: name 'results' is not defined